# Regresión logística binaria e inferencia estadística

Datos de [UCI](https://doi.org/10.24432/C55S3H), Taiwán (2005). El análisis utiliza el CSV limpio del ZIP de la auditoría de datos. El notebook puede abrirse desde la raíz del repositorio o desde `notebooks/`. Requiere pandas, numpy, scikit-learn ≥ 1.2, statsmodels, scipy y mpmath. Para cambiar la fuente por el XLS de UCI, instala también xlrd.

El análisis inferencial ajusta y describe el modelo en entrenamiento. El diagnóstico evalúa en validación. El conjunto de prueba queda reservado para las fases finales.

In [ ]:
from io import BytesIO
from pathlib import Path
import sys
from zipfile import ZipFile

import mpmath as mp
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display
from statsmodels.stats.outliers_influence import variance_inflation_factor

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))

from src.preprocessing import (
    BILLS, CATEGORICAL, PAYMENT_STATUS, load_and_split,
    make_design_pipeline, with_bill_change,
)

data_path = root / "sources" / "phase1_deliverables.zip"
splits = load_and_split(data_path)
X_train, y_train = splits["train"]
X_val, _ = splits["val"]

# Verifica train y val con sus índices archivados.
with ZipFile(data_path) as archive:
    saved_indices = archive.read("data/phase1/split_indices.npz")
with np.load(BytesIO(saved_indices)) as saved:
    for name, key in {"train": "train", "val": "validation"}.items():
        np.testing.assert_array_equal(splits[name][1].index.to_numpy(), saved[key])

analysis_splits = {name: splits[name] for name in ("train", "val")}
audit = pd.DataFrame.from_dict({
    name: {"filas": len(y), "predictores originales": X.shape[1],
           "defaults": int(y.sum()), "tasa exacta": f"{y.sum()}/{len(y)}",
           "tasa %": 100 * y.mean()}
    for name, (X, y) in analysis_splits.items()
}, orient="index")
display(audit.style.format({"tasa %": "{:.12f}"}))

## Matriz de diseño y VIF

Cada categoría se compara con el código 1 de su variable (hombre, posgrado, casado). Los estados de pago mantienen el código original, con un efecto lineal en el logit por punto de código. Los importes en NT$ y la edad en años se estandarizan. El OR de una variable continua corresponde a una desviación estándar de entrenamiento, calculada por `StandardScaler` con denominador n.

Se incluye el intercepto en las regresiones auxiliares del VIF y se reportan los VIF de los predictores.

In [2]:
def design_matrices(X_train, X_val, *, reduced_bills=False):
    """Ajusta el preprocesamiento en train y aplica la misma matriz a val."""
    pipeline = make_design_pipeline(reduced_bills=reduced_bills)
    if reduced_bills:
        X_train = with_bill_change(X_train)
        X_val = with_bill_change(X_val)
    train = sm.add_constant(pipeline.fit_transform(X_train), has_constant="add")
    val = sm.add_constant(pipeline.transform(X_val), has_constant="add")
    return pipeline, train, val


def vif_table(design):
    values = design.to_numpy()
    vif = pd.Series({
        name: variance_inflation_factor(values, position)
        for position, name in enumerate(design.columns)
        if name != "const"
    }, name="VIF")
    return vif.sort_values(ascending=False).to_frame()


def show(table):
    display(table.style.format(precision=15, formatter={
        column: "{:.16g}" for column in table.select_dtypes(include="number")
    }))


base_pipeline, base_train, base_val = design_matrices(X_train, X_val)
vif_base = vif_table(base_train)
show(vif_base.assign(**{
    "VIF > 5": vif_base["VIF"] > 5,
    "VIF > 10": vif_base["VIF"] > 10,
}))
scaler = base_pipeline.named_steps["design"].named_transformers_["continuous"]
show(pd.DataFrame({"DE train": scaler.scale_}, index=scaler.feature_names_in_))
print("Diseño base con intercepto:", base_train.shape, base_val.shape)

,VIF,VIF > 5,VIF > 10
BILL_AMT2,27.60249358316606,True,True
BILL_AMT3,25.02400839952198,True,True
BILL_AMT5,24.24948167724468,True,True
BILL_AMT4,20.71260341882105,True,True
BILL_AMT6,14.88613857171242,True,True
BILL_AMT1,13.98457726601719,True,True
PAY_5,4.664713058205288,False,False
PAY_4,4.244554129773769,False,False
PAY_3,3.64693251899597,False,False
PAY_6,3.215403514532327,False,False


,DE train
LIMIT_BAL,129763.9600682664
AGE,9.184565935650081
BILL_AMT1,73632.22909541898
BILL_AMT2,71161.00437760586
BILL_AMT3,68725.10117570653
BILL_AMT4,64427.54422286462
BILL_AMT5,61007.42100216621
BILL_AMT6,59551.82586584508
PAY_AMT1,16424.57719183207
PAY_AMT2,20701.79522535721


Diseño base con intercepto: (21000, 27) (4500, 27)


## Inferencia de máxima verosimilitud

Los errores estándar son los del modelo clásico. Los IC del 95 % son intervalos de Wald en escala logit, transformados con la exponencial. Las colas de Wald y chi-cuadrado se calculan con precisión ampliada para evitar que los p-valores muy pequeños se impriman como cero. Los p-valores son numéricos y corresponden a las aproximaciones asintóticas de sus pruebas.

In [3]:
def fit_logit(design, target):
    model = sm.Logit(target, design, missing="raise").fit(disp=False)
    if not model.mle_retvals["converged"]:
        raise RuntimeError("Logit no convergió.")
    return model


def chi_square_p(statistic, df):
    with mp.workdps(40):
        tail = mp.gammainc(mp.mpf(df) / 2, mp.mpf(statistic) / 2, mp.inf,
                           regularized=True)
        return mp.nstr(tail, 16)


def wald_p(z):
    with mp.workdps(40):
        return mp.nstr(mp.erfc(abs(mp.mpf(float(z))) / mp.sqrt(2)), 16)


def inference_table(model):
    ci = np.exp(model.conf_int())
    return pd.DataFrame({
        "beta": model.params, "EE": model.bse, "z": model.tvalues,
        "p Wald": model.tvalues.map(wald_p), "OR": np.exp(model.params),
        "OR IC95 inf": ci.iloc[:, 0], "OR IC95 sup": ci.iloc[:, 1],
    })


def fit_summary(model):
    return {
        "k": len(model.params), "Log-Likelihood": model.llf,
        "LLR p-value": chi_square_p(model.llr, int(model.df_model)),
        "AIC": model.aic, "BIC": model.bic, "Pseudo R² McFadden": model.prsquared,
    }


base = fit_logit(base_train, y_train)
base_inference = inference_table(base)
show(base_inference)
top_five = (
    base_inference.drop(index="const")
    .loc[base.pvalues.drop(index="const").lt(0.05)]
    .assign(abs_beta=lambda table: table["beta"].abs())
    .nlargest(5, "abs_beta")
    .drop(columns="abs_beta")
)
show(top_five)

,beta,EE,z,p Wald,OR,OR IC95 inf,OR IC95 sup
const,-1.144471643923439,0.04749807933602945,-24.09511415875936,2.812610972784428e-128,0.318392097756632,0.2900893880743978,0.3494561748252228
SEX_2,-0.1114946761359652,0.03681570461522009,-3.028454223578049,0.002458083145373762,0.8944961535664926,0.8322252586403308,0.961426441264799
EDUCATION_2,-0.121406165884306,0.04259753404215971,-2.850074977677057,0.004370892411770236,0.8856741559010912,0.8147322083578097,0.9627932986866938
EDUCATION_3,-0.1571863478263319,0.05705707128907649,-2.754896882631006,0.005871063584666924,0.8545448014132638,0.7641309128351104,0.9556566883455118
EDUCATION_4,-1.189989978683246,0.217854460989731,-5.462316324747366,4.699616861541845e-8,0.3042243127796306,0.198497876609294,0.4662640934361765
MARRIAGE_2,-0.1997446952424766,0.04160473382007593,-4.801008849288489,1.578682918812843e-6,0.818939805619292,0.7548105600063081,0.8885175178552867
MARRIAGE_3,-0.053559020742419,0.1528720798619879,-0.3503518810679609,0.7260746339673417,0.9478499965423798,0.7024487799654094,1.278982385006961
LIMIT_BAL,-0.110730175428426,0.02469137782407805,-4.484568508787158,7.306162992901684e-6,0.8951802579746179,0.8528902641720312,0.9395671728594952
AGE,0.03615940015540232,0.02058216477090064,1.756831730670284,0.07894649487449321,1.036821102764541,0.9958278262817298,1.079501868462301
BILL_AMT1,-0.3214121339173057,0.09541268709294938,-3.368651944622328,0.000755367494718818,0.7251243410636964,0.6014465767093387,0.874234438044139


,beta,EE,z,p Wald,OR,OR IC95 inf,OR IC95 sup
EDUCATION_4,-1.189989978683246,0.217854460989731,-5.462316324747366,4.699616861541845e-8,0.3042243127796306,0.198497876609294,0.4662640934361765
PAY_0,0.5819808831447355,0.02120022221146506,27.45164071110542,6.64122661854652e-166,1.789579870604633,1.716743549542654,1.865506420062844
BILL_AMT1,-0.3214121339173057,0.09541268709294938,-3.368651944622328,0.000755367494718818,0.7251243410636964,0.6014465767093387,0.874234438044139
PAY_AMT2,-0.254845643935462,0.05588226991274877,-4.560402509299688,5.105566173131979e-6,0.7750361202540275,0.6946321317815327,0.8647469073419342
MARRIAGE_2,-0.1997446952424766,0.04160473382007593,-4.801008849288489,1.578682918812843e-6,0.818939805619292,0.7548105600063081,0.8885175178552867


## Modelo anidado con dos términos de saldo

Se conserva el último saldo y se incluye la diferencia entre ese saldo y el promedio de los cinco anteriores. Con intercepto, centrar y estandarizar ambos términos mantiene el espacio de predictores. En la escala monetaria original, la alternativa impone que los coeficientes de `BILL_AMT2` a `BILL_AMT6` sean iguales (cuatro restricciones). La LRT contrasta esas restricciones en las mismas filas de entrenamiento.

In [4]:
fits = {"Base": fit_summary(base)}
if vif_base.loc[BILLS, "VIF"].gt(10).any():
    reduced_pipeline, reduced_train, reduced_val = design_matrices(
        X_train, X_val, reduced_bills=True,
    )
    # Comprueba que las dos matrices estén alineadas y que la reducida sea anidada.
    assert reduced_train.index.equals(base_train.index)
    full = base_train.to_numpy()
    restricted = reduced_train.to_numpy()
    mapping = np.linalg.lstsq(full, restricted, rcond=None)[0]
    np.testing.assert_allclose(full @ mapping, restricted, rtol=1e-10, atol=1e-10)
    reduced = fit_logit(reduced_train, y_train)
    fits["Reducido"] = fit_summary(reduced)
    reduced_vif = vif_table(reduced_train)
    reduced_inference = inference_table(reduced)
    show(reduced_inference)
    show(reduced_vif)
    df_difference = int(base.df_model - reduced.df_model)
    lrt = 2 * (base.llf - reduced.llf)
    show(pd.DataFrame([{"LRT": lrt, "gl": df_difference,
                        "p LRT": chi_square_p(lrt, df_difference)}]))
    print("Diseño reducido con intercepto:", reduced_train.shape, reduced_val.shape)

show(pd.DataFrame.from_dict(fits, orient="index"))

,beta,EE,z,p Wald,OR,OR IC95 inf,OR IC95 sup
const,-1.144733896969917,0.04749172866466535,-24.10385827504355,2.277373758071394e-128,0.3183086094070362,0.2900169310669922,0.3493601923511035
SEX_2,-0.1115221454482892,0.0368093437643922,-3.029723815836428,0.002447774631708329,0.8944715827097515,0.8322127734627988,0.9613880461678019
EDUCATION_2,-0.1212573867825505,0.04258025915043084,-2.847737172152076,0.004403127169872524,0.885805935509237,0.8148810224110737,0.9629039501518419
EDUCATION_3,-0.1569689140806781,0.05703871037751224,-2.751971652966471,0.005923764832573948,0.8547306284920708,0.7643245837900353,0.9558301051365121
EDUCATION_4,-1.188214043865858,0.2177679424277134,-5.456331315892733,4.860725983651023e-8,0.3047650753664168,0.1988844317338168,0.467013684044455
MARRIAGE_2,-0.1996011023516184,0.04160331890842101,-4.79772065279188,1.604813788502663e-6,0.8190574079966504,0.7549210467446699,0.88864264744905
MARRIAGE_3,-0.05305609093448359,0.1528797116723978,-0.3470446820842794,0.7285577509128492,0.9483268184530178,0.70279163876079,1.279644926030953
LIMIT_BAL,-0.1114623346081051,0.02464513552075451,-4.522691080933147,6.105832265612303e-6,0.894525083406579,0.8523432884199296,0.9387944220537107
AGE,0.03624274015739137,0.02058136200007462,1.76094955024162,0.07824694222678913,1.036907515038057,0.9959123890023571,1.07959013926862
BILL_AMT1,-0.0908341894516505,0.02992674961998932,-3.035217342513488,0.002403624065275731,0.9131691114181175,0.8611474448160485,0.968333391764613


,VIF
PAY_5,4.646966016866689
PAY_4,4.235739308122651
PAY_3,3.632223105956507
PAY_2,3.184484179563406
PAY_6,3.181287819916534
BILL_AMT1,2.145945989903855
PAY_0,1.901098623657579
BILL_AMT1_MINUS_PREV5_MEAN,1.778698207563082
LIMIT_BAL,1.579255263945144
EDUCATION_3,1.412666558967911


,LRT,gl,p LRT
0,0.4430565151342307,4,0.9788023831773322


Diseño reducido con intercepto: (21000, 23) (4500, 23)


,k,Log-Likelihood,LLR p-value,AIC,BIC,Pseudo R² McFadden
Base,27,-9708.898907248335,2.241195827369948e-574,19471.79781449667,19686.50931284772,0.1250581758157351
Reducido,23,-9709.120435505902,1.911318979674671e-578,19464.2408710118,19647.14325849603,0.1250382122400882


## Diagnóstico y validación de la especificación logística

La especificación de referencia conserva el saldo más reciente y su diferencia frente al promedio de los cinco anteriores. En la especificación inicial, imponer la igualdad de los coeficientes monetarios de esos cinco meses dio una LRT de 0.4430565 con 4 gl y p = 0.9788024. El análisis diagnóstico verifica los índices archivados de los 21.000 clientes de entrenamiento y los 4.500 de validación.

Desde la raíz, ejecuta `python -m notebooks.phase4_logistic` para reproducir las tablas de `reports/phase4_results.json` y las figuras de 300 DPI. La función `fit_phase4_logistic(X_train, y_train)` en `src/models.py` devuelve el ajuste de statsmodels y el método `predict_proba`.

- **Diagnóstico.** Se usa un GLM binomial con enlace logit, equivalente al Logit del modelo inicial. Se ordenan las filas por distancia de Cook y se reajusta sin las diez superiores, con las unidades del entrenamiento completo. El índice de cada caso es la fila del CSV limpio en base cero.
- **No linealidad.** El auxiliar de Box–Tidwell incluye conjuntamente los cuatro términos y ajusta los p-valores con Holm. Las ratios reproducen las definiciones del análisis exploratorio. Se usan utilización = saldo reciente / límite y pago = pago reciente / saldo del mes anterior, con este último denominador positivo. El término se activa en la rama positiva de cada ratio. Indicadores distinguen los ceros, la utilización negativa y el pago indefinido. Se estima con ratios crudas y se comprueba la sensibilidad a topes en los percentiles 97.5, 99 y 99.5 de los valores positivos de entrenamiento.
- **Selección.** Cada interacción conserva sus efectos principales. Se compara el BIC por log-verosimilitud dentro de 64 combinaciones de interacciones y formas para límite y edad. Cada término añadido debe alcanzar p < 0.05 en la LRT contra su omisión. Se reporta también AIC. La especificación final conserva el límite lineal, añade el logaritmo del límite y mantiene las interacciones de `PAY_0` con límite y con saldo reciente.
- **Validación.** Los diez pliegues estratificados ajustan su propio preprocesamiento y repiten la selección. Se presenta también el desempeño condicional a la fórmula final. PR-AUC se estima como average precision y la desviación estándar es muestral. El umbral de Youden se aprende de las predicciones fuera de pliegue de entrenamiento. El máximo calculado en validación se muestra como referencia descriptiva del mismo conjunto.

El modelo final obtuvo AIC 19171.936489 y BIC 19378.695710. Los diez pliegues eligieron la misma fórmula. En validación, ROC-AUC = 0.71908042, average precision = 0.49897516, Log Loss = 0.46309304 y Brier = 0.14422851. El umbral de Youden aprendido en entrenamiento (0.25906180) detectó 523 de 995 incumplimientos en validación, frente a los 285 detectados con 0.50. Los falsos positivos aumentaron de 136 a 556.

De las 21.000 filas de entrenamiento, 831 superaron |d| > 2 y 13 superaron |d| > 3. Los diez casos con mayor distancia de Cook fueron incumplimientos con probabilidades ajustadas inferiores a 0.10. Al excluirlos, el mayor cambio relativo en un OR fue de −5.27 % en PAY_AMT2. PAY_AMT4 pasó de p = 0.06660 a p = 0.00508. La interacción de persistencia PAY_0 × PAY_2 dio p = 0.401472 al añadirla al modelo lineal que ya incorporaba interacciones con límite y saldo.

In [ ]:
import json
from IPython.display import Image

phase4 = json.loads((root / "reports" / "phase4_results.json").read_text(encoding="utf-8"))

display(Image(filename=str(root / "figures" / "05_logistic_diagnostics_residuals.png"), width=920))
display(Image(filename=str(root / "figures" / "06_logistic_influence_cooks.png"), width=920))
for item in ("devianza", "top10", "sensibilidad"):
    show(pd.DataFrame(phase4["diagnósticos"][item]))

show(pd.DataFrame(phase4["Box–Tidwell"]["dominios"]).T)
for fit in phase4["Box–Tidwell"]["ajustes"]:
    print("Cuantil del tope", fit["cuantil_tope"], fit["topes"])
    show(pd.DataFrame(fit["términos"]))

show(pd.DataFrame(phase4["comparación"]["modelos"]))
show(pd.DataFrame(phase4["comparación"]["omisión individual del final"]))
show(pd.DataFrame(phase4["inferencia final"]).T)
show(pd.DataFrame(phase4["VIF saldos"]))
show(pd.DataFrame(phase4["CV"]["resumen"]).T)
show(pd.DataFrame(phase4["CV"]["resumen con especificación fija"]).T)
show(pd.Series(phase4["val"]["probabilísticas"], name="val").to_frame())
show(pd.DataFrame(phase4["val"]["umbrales"]).T)

### Limitaciones del enfoque paramétrico

- Los residuos de una respuesta Bernoulli siguen ramas separadas por clase. Las referencias de devianza, apalancamiento y Cook son diagnósticas. La exclusión de diez filas evalúa la estabilidad local y la estimación final usa todos los clientes de entrenamiento.
- `PAY_0` y los otros estados de pago conservan su codificación ordinal. A límite y saldo fijos, este supuesto impone un incremento constante del logit por punto de código para PAY_0. Los demás estados de pago conservan pendientes lineales sin interacciones. Los códigos de ausencia de consumo, pago al día y atraso pueden representar estados cualitativamente distintos. La interacción y la curvatura del límite amplían la forma del logit sin modelar todas las discontinuidades posibles.
- La curvatura detectada para utilización se refiere al tramo positivo. El resultado para la razón de pago depende del extremo superior, donde saldos previos muy pequeños producen cocientes grandes. La utilización justifica explorar splines por tramos en otro modelo paramétrico. El modelo final usa el saldo reciente para representar la carga financiera en la interacción.
- Los efectos principales de `LIMIT_BAL` y su logaritmo se interpretan en conjunto. Para un cambio real de límite a mora fija cambian ambos términos y la interacción. Los OR de los términos de interacción multiplican OR condicionales por unidad del producto. Los intervalos de Wald se calculan condicionales a la especificación seleccionada y omiten la incertidumbre del proceso de selección.
- Los cortes maximizan sensibilidad más especificidad menos uno. Una decisión crediticia requeriría costes de decisión, validación externa y evaluación de calibración. El Brier y el Log Loss miden la calidad probabilística global y la calibración se estudiará en el análisis de calibración.
- Los coeficientes describen asociaciones observacionales en clientes de tarjetas de crédito de Taiwán en 2005. La inferencia causal requeriría un diseño que identificara efectos. Su aplicación en Colombia requiere validación y calibración con datos locales.